# RAG with LangChain

Author: Umberto Michelucci, umberto.michelucci@hslu.ch

In this notebook we build a minimal Retrieval-Augmented Generation (RAG) system using LangChain.

The system:
1. Reads PDF documents
2. Splits them into chunks
3. Converts chunks into embeddings
4. Stores embeddings in a vector database
5. Retrieves the most relevant chunks
6. Uses an LLM to answer questions

This architecture is the basis of many modern AI assistants.

## What is different from the previous implementation?

In the previous notebook we implemented everything manually using:
- plain Python
- NumPy
- custom cosine similarity code

That approach was useful because it showed the internal mechanics of RAG very clearly.

We manually implemented:
- PDF loading
- chunking
- embeddings
- similarity search
- retrieval
- prompt construction

In this notebook, LangChain automates many of these steps.

LangChain provides:
- document loaders
- text splitters
- vector stores
- retrievers
- integrations with LLMs

This makes the code:
- shorter
- more modular
- easier to scale

However, the core ideas are EXACTLY the same.

In [1]:
# Install required packages
!pip install -q langchain langchain-openai langchain-community langchain-text-splitters pypdf


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# Import libraries
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

/Users/umbertomichelucci/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from openai import OpenAI
from pathlib import Path
import numpy as np
from pypdf import PdfReader

# Path to the file on the Desktop
key_path = Path.home() / "Desktop" / "api-key.txt"

# Read the key
api_key = key_path.read_text().strip()

## Step 1 — Load PDF documents

We first load the PDF files.

LangChain converts each PDF page into a `Document` object.

Each `Document` contains:
- the page text
- metadata

The metadata includes information such as:
- source file name
- page number

This is very useful because it allows us to trace where the answer comes from.

In [4]:
# Names of your two PDF files
# In Colab, upload these files first using the file browser or files.upload()
pdf_files = [
    "paper1.pdf",
    "paper2.pdf"
]

In [5]:
# Load the PDFs
# Each page becomes a LangChain Document.
# Metadata automatically contains information such as source file and page number.

docs = []

for pdf_file in pdf_files:
    loader = PyPDFLoader(pdf_file)
    pages = loader.load()
    docs.extend(pages)

print("Number of loaded pages:", len(docs))

Number of loaded pages: 16


## Step 2 — Split documents into chunks

Large PDFs cannot be embedded directly.

We therefore split the text into smaller chunks.

Why chunking is important:
- embeddings have size limits
- retrieval works better on smaller semantic units
- smaller chunks improve precision

LangChain provides built-in text splitters that automate this process.

In [6]:
# Split the pages into smaller chunks
# This is necessary because PDFs are too long to embed or send to the LLM directly.

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

splits = text_splitter.split_documents(docs)

print("Number of chunks:", len(splits))

Number of chunks: 117


## Step 3 — Create embeddings

Embeddings convert text into vectors of numbers.

Texts with similar meaning produce similar vectors.

Example:
- "heart disease"
- "cardiovascular disease"

These texts will produce nearby vectors in embedding space.

The embedding model does NOT answer questions.
It only creates numerical representations of text.

In [7]:
# Create embeddings
# Embeddings transform text into numerical vectors.

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=api_key
)

## Step 4 — Create a vector store

The vector store stores:
- text chunks
- embeddings
- metadata

This allows fast semantic retrieval.

In this notebook we use:
- `InMemoryVectorStore`

because it is simple and works well for teaching.

In production systems people often use:
- FAISS
- Chroma
- Pinecone
- Weaviate
- Milvus

In [8]:
# Create an in-memory vector store
# This stores the chunks and their embeddings.
# For teaching, this is simpler than using Chroma or FAISS.

vectorstore = InMemoryVectorStore.from_documents(
    documents=splits,
    embedding=embeddings
)

## Step 5 — Retrieval

When the user asks a question:

1. The question is embedded
2. The retriever compares the question embedding with document embeddings
3. The most relevant chunks are returned

This is semantic search.

Unlike keyword search, semantic search finds meaning, not exact words.

In [9]:
# Create a retriever
# The retriever searches for the most relevant chunks for a question.

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [10]:
# Create the language model
# This model will generate the final answer using retrieved chunks.

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    api_key=api_key
)

In [11]:
# Helper function to format retrieved documents as context

def format_context(retrieved_docs):
    context = ""

    for i, doc in enumerate(retrieved_docs, start=1):
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page", "unknown")

        context += f"""
Source {i}
Document: {source}
Page: {page}
Text:
{doc.page_content}
"""

    return context

## Why RAG is powerful

RAG allows LLMs to:
- use external knowledge
- access private documents
- reduce hallucinations
- answer questions about specific PDFs or databases

Without RAG:
- the model only knows its training data

With RAG:
- the model can use new and domain-specific information.

In [12]:
# Main RAG function

def rag_answer(question):

    # 1. Retrieve relevant chunks from the PDFs
    retrieved_docs = retriever.invoke(question)

    # 2. Convert retrieved chunks into text context
    context = format_context(retrieved_docs)

    # 3. Build the prompt
    prompt = f"""
Answer the question using ONLY the context below.

At the end of the answer, cite the document and page number.

Context:
{context}

Question:
{question}
"""

    # 4. Ask the LLM
    response = llm.invoke(prompt)

    # 5. Print retrieved sources
    print("=== RETRIEVED SOURCES ===\n")

    for i, doc in enumerate(retrieved_docs, start=1):
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page", "unknown")

        print(f"Source {i}")
        print("Document:", source)
        print("Page:", page)
        print("Text preview:", doc.page_content[:400], "...")
        print()

    # 6. Print final answer
    print("=== ANSWER ===\n")
    print(response.content)

In [13]:
# Try it

question = "What is the leading cause of cardiovascular disease?"

rag_answer(question)

=== RETRIEVED SOURCES ===

Source 1
Document: paper2.pdf
Page: 0
Text preview: Hamilton, Ontario L8P 1H1. Telephone 905-523-7284 ext 5282, fax 905-522-0568, e-mail tarride@mcmaster.ca
Received for publication January 16, 2007. Accepted November 18, 2007
C
ardiovascular disease (CVD) is one of the leading causes of global 
mortality and morbidity, and is responsible for an estimated 
16.7 million deaths worldwide (30% of all deaths) (1). In North 
America, 74,255 deaths (33%  ...

Source 2
Document: paper1.pdf
Page: 5
Text preview: lence and slower progression of coronary artery calcification.
Among men in a post–World War II birth cohort, Japanese
Americans and whites had a similarly high prevalence of
coronary artery calcification that was significantly higher
than that in Japanese people living in Japan.
64
Conclusions
In Asian countries, stroke is more prominent than CHD. This
is most likely due to a higher prevalence of ...

Source 3
Document: paper1.pdf
Page: 5
Text preview: impor

## Manual RAG vs LangChain RAG

| Feature | Manual Implementation | LangChain |
|---|---|---|
| Educational value | Very high | Medium |
| Amount of code | Larger | Smaller |
| Transparency | Full control | More abstraction |
| Ease of scaling | Harder | Easier |
| Production readiness | Low | Higher |
| Flexibility | High | High |
| Ease for beginners | Medium | High |

The manual implementation is excellent for understanding how RAG works internally.

LangChain is useful when building larger applications because it automates many common tasks.